# Avaliação — APIs de energia renovável e aprendizado de máquina

Consulte duas APIs públicas, organize os dados e resolva **duas tarefas**: classificar a fonte de empreendimentos da ANEEL e estimar a radiação solar em Petrolina (PE). Em **cada tarefa**, treine e compare **três algoritmos diferentes**. Este notebook fornece a consulta às APIs e os arquivos de dados; **você implementará a análise e os modelos**.

**Entrega:** um único link para um repositório público no GitHub com notebook executável, arquivos de dados, README e resultados. As instruções completas estão ao final e no enunciado separado.

## O que é recebido da API?

Uma requisição HTTP `GET` pede dados a uma URL. Os parâmetros identificam o recurso, o período e as variáveis. A resposta é **JSON**: um objeto com chaves e valores. Nas duas consultas desta avaliação, o acesso público **não exige login, token ou chave de API**. Se outra API exigir credenciais, não as publique no repositório.

Este código usa somente `json`, `urllib` e `csv`, bibliotecas que vêm com Python. `csv` serve apenas para exportar os dados recebidos; os imports para análise, tabelas, gráficos e ML são parte da sua implementação.

In [ ]:
import json
import urllib.parse
import urllib.request
import csv

A função a seguir monta a URL, abre a conexão, verifica o status HTTP e converte o JSON. Se houver falha na rede ou na API, exibe uma mensagem de erro.

In [ ]:
def consultar_api(endereco, parametros):
    url = endereco + "?" + urllib.parse.urlencode(parametros)
    try:
        with urllib.request.urlopen(url, timeout=30) as resposta:
            return json.load(resposta)
    except Exception as erro:
        raise RuntimeError(f"Não foi possível consultar a API: {erro}") from erro

## Bibliotecas de análise e aprendizado de máquina

Imports acrescentados para a análise (não fazem parte do notebook de apoio). A semente `SEMENTE = 42` é usada em todas as divisões e modelos com aleatoriedade.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              HistGradientBoostingRegressor)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             mean_absolute_error, mean_squared_error, r2_score)

SEMENTE = 42
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")

# Tarefa 1 — Classificação (ANEEL)

**Pergunta:** a partir da potência e da localização de um empreendimento, é possível classificar sua fonte como **Solar, Eólica ou Hidráulica**?

A fonte é uma **categoria**: o problema é de classificação. O conjunto [SIGA da ANEEL](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel) registra empreendimentos em várias fases. Usamos `UFV` (solar fotovoltaica), `EOL` (eólica) e três categorias hidráulicas: `UHE`, `PCH`, `CGH`. Estas três últimas formarão uma única classe, *Hidráulica*. A tarefa trata da **categoria do cadastro**, não da quantidade de energia gerada.

### Atributos originais da resposta

| Campo da API | Significado | Uso na tarefa |
|---|---|---|
| `SigTipoGeracao` | Sigla do tipo de geração (`UFV`, `EOL`, `UHE`, `PCH`, `CGH`) | Origina o alvo `fonte`; **não entra em X** |
| `MdaPotenciaOutorgadaKw` | Potência outorgada do empreendimento, em quilowatts (kW) | Entrada `potencia_kw`; potência nominal autorizada, não energia produzida |
| `NumCoordNEmpreendimento` | Latitude aproximada do empreendimento, em graus decimais | Entrada `latitude` |
| `NumCoordEEmpreendimento` | Longitude aproximada do empreendimento, em graus decimais | Entrada `longitude` |
| `CodCEG`, `NomEmpreendimento` e campos de combustível | Código, nome e descrição associados ao empreendimento e à fonte | **Não usar como entradas**: podem revelar a classe |

A resposta CKAN contém `success` e `result`; dentro de `result`, `records` são as linhas, `fields` descrevem as colunas e `total` indica o total encontrado para o filtro. O CSV final terá **três atributos numéricos e a classe**. Quantidades consultadas têm limite por tipo e não representam a participação de cada fonte na matriz brasileira.

### 1. Consultar a ANEEL

O recurso SIGA é acessado pela API pública CKAN/DataStore. `resource_id` aponta para o conjunto; `filters` seleciona uma sigla; `limit` limita o retorno de cada pedido. Não há etapa de autenticação para esta consulta pública.

In [ ]:
URL_ANEEL = "https://dadosabertos.aneel.gov.br/api/3/action/datastore_search"
ID_RECURSO = "11ec447d-698d-4ab8-977f-b424d5deee6a"
SIGLAS = ["UFV", "EOL", "UHE", "PCH", "CGH"]
print("API ANEEL: consulta pública, sem token")

In [ ]:
registros_aneel = []
for sigla in SIGLAS:
    parametros = {
        "resource_id": ID_RECURSO,
        "filters": json.dumps({"SigTipoGeracao": sigla}),
        "limit": 1200,
    }
    resposta = consultar_api(URL_ANEEL, parametros)
    if not resposta.get("success"):
        raise ValueError(f"Consulta da ANEEL falhou para {sigla}")
    linhas = resposta["result"]["records"]
    print(f"{sigla}: {len(linhas)} linhas recebidas")
    registros_aneel.extend(linhas)
print("Total de linhas recebidas:", len(registros_aneel))

### 2. Gerar o CSV para a tarefa

A API pode fornecer números como texto, inclusive com vírgula decimal. O tratamento abaixo converte esses valores, descarta linhas sem potência ou coordenadas e transforma as siglas em três classes. Cada linha do arquivo final representa **um empreendimento**. Confira a quantidade de exemplos por classe durante sua análise; se necessário, discuta o desequilíbrio entre classes.

In [ ]:
def numero(valor):
    texto = str(valor).strip()
    if "," in texto:
        texto = texto.replace(".", "").replace(",", ".")
    try:
        return float(texto)
    except ValueError:
        return None

fontes = {"UFV": "Solar", "EOL": "Eólica", "UHE": "Hidráulica",
          "PCH": "Hidráulica", "CGH": "Hidráulica"}
linhas_classificacao = []
for registro in registros_aneel:
    potencia = numero(registro.get("MdaPotenciaOutorgadaKw"))
    latitude = numero(registro.get("NumCoordNEmpreendimento"))
    longitude = numero(registro.get("NumCoordEEmpreendimento"))
    fonte = fontes.get(registro.get("SigTipoGeracao"))
    if fonte and all(valor is not None for valor in (potencia, latitude, longitude)):
        linhas_classificacao.append([potencia, latitude, longitude, fonte])
if len(linhas_classificacao) < 90:
    raise ValueError("Poucos registros válidos: confira a resposta da ANEEL")
print("Linhas válidas:", len(linhas_classificacao))
print("Primeiro exemplo:", linhas_classificacao[0])

In [ ]:
with open("aneel_classificacao_orange.csv", "w", newline="", encoding="utf-8-sig") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["potencia_kw", "latitude", "longitude", "fonte"])
    escritor.writerows(linhas_classificacao)
print("Salvo: aneel_classificacao_orange.csv")

### O que você deve fazer — classificação

1. Carregue o CSV, confira os tipos das colunas, as contagens por classe, valores ausentes e distribuições relevantes. Explique `X` (três entradas) e `y` (`fonte`).
2. Faça uma divisão **estratificada** de treino/teste, por exemplo 80%/20%, com semente fixa. Ajustes de escala aprendidos dos dados devem ser feitos **somente com o treino**.
3. Escolha, justifique e treine **três classificadores diferentes** usando a mesma divisão. Eles **não estão importados nem implementados** neste notebook.
4. Compare Accuracy, Precision, Recall e F1. Declare o tipo de média usado nas métricas por classe (por exemplo, `macro`) e apresente a matriz de confusão.
5. Explique qual modelo escolheria, quais classes ele mais confunde e por que potência/localização podem não bastar para uma aplicação real. Não use `SigTipoGeracao`, nomes, CEG ou descrições da fonte como entradas.

**O que registrar:** tabela comparativa dos três algoritmos, gráfico(s) ou matriz de confusão e interpretação escrita.

## Solução da Tarefa 1 — Classificação da fonte

### 1.1 Carregar e explorar o CSV

In [ ]:
aneel = pd.read_csv("aneel_classificacao_orange.csv", encoding="utf-8-sig")
print("Linhas e colunas:", aneel.shape)
print("\nTipos das colunas:")
print(aneel.dtypes)
print("\nValores ausentes por coluna:")
print(aneel.isna().sum())
print("\nLinhas duplicadas:", aneel.duplicated().sum())
aneel.head()

In [ ]:
contagem = aneel["fonte"].value_counts()
distribuicao = pd.DataFrame({"empreendimentos": contagem,
                             "proporcao_%": 100 * contagem / contagem.sum()})
distribuicao

In [ ]:
aneel.groupby("fonte")[["potencia_kw", "latitude", "longitude"]].describe().T

A potência outorgada é muito assimétrica (de poucos kW a milhões de kW), por isso os gráficos usam escala logarítmica. O mapa usa latitude × longitude apenas como dispersão, sem projeção cartográfica.

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(17, 4.8))
sns.countplot(data=aneel, x="fonte", order=contagem.index, ax=eixos[0])
eixos[0].set_title("Empreendimentos por classe")
eixos[0].set_xlabel("")

sns.boxplot(data=aneel, x="fonte", y="potencia_kw", order=contagem.index, ax=eixos[1])
eixos[1].set_yscale("log")
eixos[1].set_title("Potência outorgada por classe (escala log)")
eixos[1].set_xlabel("")
eixos[1].set_ylabel("potência (kW)")

sns.scatterplot(data=aneel, x="longitude", y="latitude", hue="fonte",
                hue_order=contagem.index, s=10, alpha=0.6, ax=eixos[2])
eixos[2].set_title("Localização aproximada dos empreendimentos")
plt.tight_layout()
plt.show()

In [ ]:
# Coordenadas fora de um retângulo aproximado do território brasileiro
fora_brasil = ~aneel["latitude"].between(-34, 6) | ~aneel["longitude"].between(-74, -34)
print("Registros com coordenadas fora do retângulo do Brasil:", fora_brasil.sum())
print("Registros com potência <= 0:", (aneel["potencia_kw"] <= 0).sum())

**O que a exploração mostra**

- **3.876 empreendimentos, sem valores ausentes** nas quatro colunas: o código de apoio já descarta linhas sem potência ou coordenadas. Há 28 linhas duplicadas (mesma potência, coordenadas e fonte). Elas foram mantidas porque o CSV não tem identificador que permita saber se são o mesmo empreendimento repetido.
- **Classes:** Hidráulica 1.476 (38%), Solar 1.200 (31%) e Eólica 1.200 (31%) — um desequilíbrio leve. As quantidades são **limitadas pela consulta** (até 1.200 por sigla; a API informa 18.980 registros UFV e 1.569 EOL no total). Por isso **não representam a participação das fontes na matriz brasileira**. Os registros hidráulicos (UHE 221 + PCH 537 + CGH 718) vieram completos.
- **Potência:** as eólicas se concentram entre ~23 e 37 MW (mediana de 29,6 MW). As hidráulicas vão de CGHs com menos de 1 MW a UHEs de mais de 11 GW. Na amostra solar, **753 das 1.200 usinas têm exatamente 1 kW**.
- **Localização:** eólicas no Nordeste e no litoral Sul (longitude mediana ≈ −40,7°); hidráulicas mais ao Sul/Sudeste (latitude mediana ≈ −22°). A maior parte das solares de 1 kW está em um único aglomerado próximo de (−2°, −53°), na Região Norte.
- **Qualidade:** 47 registros (24 eólicos e 23 hidráulicos) têm latitude ou longitude igual a 0, ou seja, coordenada ausente gravada como zero. Eles foram mantidos para respeitar o CSV do notebook de apoio, e o efeito deles é medido na verificação de sensibilidade abaixo.

### 1.2 Definir `X` e `y` e separar treino/teste

- `X` = `potencia_kw`, `latitude`, `longitude` (três entradas numéricas).
- `y` = `fonte` (Solar, Eólica ou Hidráulica).

Nenhum nome, código CEG, sigla (`SigTipoGeracao`) ou descrição entra em `X`: o CSV contém apenas os três atributos e a classe. A divisão é **estratificada 80%/20%** com `random_state=42`, e os três classificadores usam exatamente os mesmos conjuntos de treino e teste.

In [ ]:
X_clf = aneel[["potencia_kw", "latitude", "longitude"]]
y_clf = aneel["fonte"]

X_treino_c, X_teste_c, y_treino_c, y_teste_c = train_test_split(
    X_clf, y_clf, test_size=0.20, stratify=y_clf, random_state=SEMENTE)

print("Treino:", X_treino_c.shape, " Teste:", X_teste_c.shape)
pd.DataFrame({"treino_%": 100 * y_treino_c.value_counts(normalize=True),
              "teste_%": 100 * y_teste_c.value_counts(normalize=True)})

### 1.3 Três classificadores

| Modelo | Por que foi escolhido | Pré-processamento |
|---|---|---|
| **Regressão Logística** (multinomial) | Modelo linear, simples e interpretável; serve de referência para fronteiras lineares | `log1p` na potência + `StandardScaler` |
| **KNN** (k = 15, pesos por distância) | Classifica pela vizinhança: empreendimentos próximos no espaço potência/localização tendem a ser da mesma fonte | `log1p` na potência + `StandardScaler` (distâncias exigem escalas comparáveis) |
| **Random Forest** (300 árvores) | Captura fronteiras não lineares e interações (ex.: região × faixa de potência) sem exigir escala | Nenhum |

A transformação `log1p` e a padronização ficam **dentro do `Pipeline`**, portanto a média e o desvio-padrão são aprendidos **somente com o treino** e apenas aplicados ao teste. Um `DummyClassifier` (sempre prevê a classe mais frequente) é mostrado só como linha de base; ele não é um dos três modelos comparados.

In [ ]:
def log_potencia(X):
    X = np.asarray(X, dtype=float).copy()
    X[:, 0] = np.log1p(np.clip(X[:, 0], 0, None))  # coluna 0 = potencia_kw
    return X

classificadores = {
    "Regressão Logística": make_pipeline(
        FunctionTransformer(log_potencia), StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=SEMENTE)),
    "KNN (k=15)": make_pipeline(
        FunctionTransformer(log_potencia), StandardScaler(),
        KNeighborsClassifier(n_neighbors=15, weights="distance")),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, random_state=SEMENTE, n_jobs=-1),
}

linha_base = DummyClassifier(strategy="most_frequent").fit(X_treino_c, y_treino_c)
print("Linha de base (classe mais frequente) — Accuracy no teste:",
      round(accuracy_score(y_teste_c, linha_base.predict(X_teste_c)), 3))

In [ ]:
resultados_clf = []
previsoes_clf = {}
for nome, modelo in classificadores.items():
    modelo.fit(X_treino_c, y_treino_c)
    previsto = modelo.predict(X_teste_c)
    previsoes_clf[nome] = previsto
    resultados_clf.append({
        "Algoritmo": nome,
        "Accuracy": accuracy_score(y_teste_c, previsto),
        "Precision (macro)": precision_score(y_teste_c, previsto, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_teste_c, previsto, average="macro", zero_division=0),
        "F1 (macro)": f1_score(y_teste_c, previsto, average="macro", zero_division=0),
        "F1 (weighted)": f1_score(y_teste_c, previsto, average="weighted", zero_division=0),
    })

tabela_clf = pd.DataFrame(resultados_clf).set_index("Algoritmo").sort_values("F1 (macro)", ascending=False)
print("Avaliação: holdout estratificado 80/20, random_state=42, mesmo conjunto de teste para os três modelos")
print(f"Treino = {len(X_treino_c)} empreendimentos | Teste = {len(X_teste_c)} empreendimentos")
tabela_clf

**Tipo de média:** Precision, Recall e F1 usam **média `macro`** (média simples das três classes), para que a classe minoritária pese tanto quanto as majoritárias. O F1 `weighted` (ponderado pelo número de exemplos de cada classe) é mostrado apenas como complemento.

In [ ]:
classes = list(contagem.index)
fig, eixos = plt.subplots(1, 3, figsize=(18, 5))
for eixo, (nome, previsto) in zip(eixos, previsoes_clf.items()):
    matriz = confusion_matrix(y_teste_c, previsto, labels=classes)
    ConfusionMatrixDisplay(matriz, display_labels=classes).plot(ax=eixo, cmap="Blues", colorbar=False)
    eixo.set_title(nome)
    eixo.set_xlabel("Classe prevista")
    eixo.set_ylabel("Classe real")
plt.suptitle("Matrizes de confusão no conjunto de teste (linhas = real, colunas = previsto)")
plt.tight_layout()
plt.show()

In [ ]:
melhor_clf = tabela_clf.index[0]
print("Relatório por classe —", melhor_clf)
print(classification_report(y_teste_c, previsoes_clf[melhor_clf], digits=3, zero_division=0))

# Pares (real -> previsto) mais frequentes entre os erros do melhor modelo
erros = pd.DataFrame({"real": y_teste_c.values, "previsto": previsoes_clf[melhor_clf]})
erros = erros[erros["real"] != erros["previsto"]]
erros.value_counts().rename("quantidade").to_frame()

In [ ]:
importancias = pd.Series(classificadores["Random Forest"].feature_importances_,
                         index=X_clf.columns).sort_values(ascending=False)
print("Importância das variáveis (Random Forest, redução média de impureza):")
importancias.to_frame("importância")

### Verificação de sensibilidade da amostra

A API devolve no máximo 1.200 registros por sigla, na ordem do cadastro, e **não uma amostra aleatória**. Na exploração aparecem dois grupos atípicos: usinas solares com potência de exatamente 1 kW concentradas em uma mesma região, e empreendimentos com latitude ou longitude igual a 0 (coordenada ausente registrada como zero). A célula abaixo **não altera o treino**: ela apenas separa o **mesmo conjunto de teste** entre registros típicos e atípicos para medir quanto esses grupos influenciam as métricas.

In [ ]:
atipico = ((X_teste_c["potencia_kw"] == 1) | (X_teste_c["latitude"] == 0)
           | (X_teste_c["longitude"] == 0)).values
print("Teste: típicos =", (~atipico).sum(), "| atípicos =", atipico.sum())
print("Classes entre os típicos:", y_teste_c[~atipico].value_counts().to_dict())

sensibilidade = []
for nome, previsto in previsoes_clf.items():
    for grupo, mascara in [("típicos", ~atipico), ("atípicos (1 kW ou coordenada 0)", atipico)]:
        sensibilidade.append({"Algoritmo": nome, "Subconjunto do teste": grupo,
                              "Accuracy": accuracy_score(y_teste_c[mascara], previsto[mascara]),
                              "F1 (macro)": f1_score(y_teste_c[mascara], previsto[mascara],
                                                     average="macro", zero_division=0)})
pd.DataFrame(sensibilidade).set_index(["Algoritmo", "Subconjunto do teste"])

### 1.4 Interpretação — classificação

**Comparação.** Com a mesma divisão estratificada 80/20 (`random_state=42`, 3.100 exemplos de treino e 776 de teste) e métricas com **média macro**:

| Algoritmo | Accuracy | Precision (macro) | Recall (macro) | F1 (macro) |
|---|---|---|---|---|
| Random Forest | 0,972 | 0,973 | 0,970 | 0,971 |
| KNN (k = 15) | 0,972 | 0,972 | 0,970 | 0,971 |
| Regressão Logística | 0,834 | 0,850 | 0,830 | 0,829 |

A linha de base (sempre "Hidráulica") acerta 38%. Os três modelos aprendem algo útil, mas **Random Forest e KNN empatam** e superam a Regressão Logística por cerca de 14 pontos de F1 macro.

**Modelo escolhido: Random Forest.** Ele empata com o KNN no teste, mas não depende de escolher uma métrica de distância nem de escala. Também indica a importância das variáveis: latitude 0,38, potência 0,31 e longitude 0,30, ou seja, as três contribuem. O KNN seria uma alternativa igualmente válida.

**Classes mais confundidas.**
- **Solar** é a classe com menor recall (0,94 no Random Forest), e é a que mais aparece entre os erros do modelo:
  - **Solar → Eólica (8 casos):** todas são usinas solares de 18 a 48 MW no interior do Nordeste (latitudes −5° a −14°, longitudes −36° a −42°). É exatamente a faixa de potência e a região típicas dos parques eólicos.
  - **Solar → Hidráulica (7 casos):** usinas solares de poucos kW a 9 MW no Sudeste/Sul e no Centro-Oeste (MG, SP, RJ, RS, MT), regiões e portes em que predominam PCHs e CGHs.
- **Eólica → Hidráulica (4 casos):** eólicas pequenas (2 a 4 MW) ou localizadas fora do padrão (Sudeste/Sul). **Hidráulica → Eólica (2 casos):** pequenas hidrelétricas no litoral nordestino.
- Na **Regressão Logística** a confusão dominante é **Solar → Eólica (62 casos)**. Uma fronteira linear em potência/lat/long não separa bem solares grandes do Nordeste de eólicas da mesma região e faixa de potência.

**Cuidado com o otimismo das métricas.** A verificação de sensibilidade mostra que os registros atípicos (1 kW ou coordenada 0) são acertados em 97–100% dos casos. Eles não refletem uma capacidade geral do modelo: vêm da forma como a amostra foi obtida, com as primeiras 1.200 usinas solares do cadastro concentradas em um aglomerado de 1 kW. Nos registros típicos, o F1 macro cai para ≈ 0,95 no Random Forest e no KNN e para ≈ 0,60 na Regressão Logística. Com uma amostra aleatória das 18.980 usinas solares, o desempenho provavelmente seria menor.

**Limitações de prever a fonte só por potência e localização.**
- As mesmas regiões recebem várias fontes: o Nordeste tem eólicas, solares e PCHs, e o Sul tem hidráulicas e eólicas.
- As faixas de potência se sobrepõem (usinas de 5–50 MW existem nas três fontes).
- O cadastro mistura fases diferentes (em operação, em construção, outorgadas) e tem coordenadas aproximadas ou ausentes.
- Por isso o modelo aprende **padrões geográficos da amostra consultada**, não uma característica física da tecnologia. Na prática, a fonte de um empreendimento é informação cadastral conhecida; um classificador assim só serve para exercício ou triagem de dados inconsistentes, e precisaria de atributos físicos (recurso solar/eólico local, relevo, bacia hidrográfica, fator de capacidade) para generalizar.

# Tarefa 2 — Regressão (Open-Meteo)

**Pergunta:** dadas as condições meteorológicas e a hora local em Petrolina (PE), qual é a radiação solar horizontal média naquela hora? A resposta é um **número**, então o problema é de regressão.

A API histórica do [Open-Meteo](https://open-meteo.com/en/docs/historical-weather-api) devolve dados horários para coordenadas e período escolhidos. Usamos **1º de abril a 30 de junho de 2025**, em horário local. São dados históricos estimados por modelos/reanálise, não leituras de um sensor particular. **Radiação em W/m² não equivale à energia em kWh nem à geração de um painel.**

### Parâmetros da requisição

| Parâmetro | Valor nesta avaliação | Significado |
|---|---|---|
| `latitude`, `longitude` | `-9.39`, `-40.50` | Coordenadas aproximadas de Petrolina (PE) |
| `start_date`, `end_date` | `2025-04-01`, `2025-06-30` | Período histórico consultado, inclusive as datas extremas |
| `timezone` | `America/Recife` | Faz `time` representar a hora local |
| `hourly` | Lista das cinco variáveis abaixo | Solicita valores horários em JSON |

### Atributos do conjunto de dados

| Campo da API → CSV | Significado e unidade | Uso na tarefa |
|---|---|---|
| `time` → `data_hora` | Data e hora local de cada registro | Ordenar e separar por tempo; não é entrada em X |
| `temperature_2m` → `temperatura_c` | Temperatura do ar a 2 m, em °C | Entrada |
| `relative_humidity_2m` → `umidade_pct` | Umidade relativa a 2 m, em % | Entrada |
| `cloud_cover` → `nuvens_pct` | Cobertura total de nuvens, em % | Entrada |
| `wind_speed_10m` → `vento_kmh` | Velocidade do vento a 10 m, em km/h | Entrada |
| `time` → `hora` | Hora local extraída da data, de 7 a 17 nesta amostra | Entrada |
| `shortwave_radiation` → `radiacao_w_m2` | Radiação solar global horizontal média da hora anterior, em W/m² | Alvo numérico `y`; **não entra em X** |

A resposta inclui `hourly` (listas dos valores, alinhadas por índice), `hourly_units` (unidades) e metadados da localização. Cada linha do CSV final representa **uma hora entre 7h e 17h**; esse recorte reduz o domínio de horas noturnas, mas não remove a importância da hora para a previsão.

### 3. Consultar o histórico meteorológico

A chamada usa GET, sem cadastro ou chave para esta consulta pública de uso educacional. Se `hourly` faltar, verifique datas e parâmetros.

In [ ]:
URL_METEO = "https://archive-api.open-meteo.com/v1/archive"
parametros_meteo = {
    "latitude": -9.39,
    "longitude": -40.50,
    "start_date": "2025-04-01",
    "end_date": "2025-06-30",
    "timezone": "America/Recife",
    "hourly": ",".join(["temperature_2m", "relative_humidity_2m",
                         "cloud_cover", "wind_speed_10m", "shortwave_radiation"]),
}
print("Open-Meteo: consulta pública, sem token")

In [ ]:
resposta_meteo = consultar_api(URL_METEO, parametros_meteo)
if "hourly" not in resposta_meteo:
    raise ValueError("Resposta sem a seção hourly")
horas_api = resposta_meteo["hourly"]
print("Campos recebidos:", list(horas_api))
print("Unidades informadas pela API:", resposta_meteo.get("hourly_units", {}))
print("Quantidade de horários:", len(horas_api["time"]))

### 4. Gerar o CSV horário

As listas de `hourly` são alinhadas: o índice 0 de cada variável se refere ao índice 0 de `time`. Percorremos esses índices, mantemos as horas de 7h a 17h e descartamos linhas com valores ausentes. A data permanece ordenada para facilitar uma avaliação temporal.

In [ ]:
nomes_api = ["temperature_2m", "relative_humidity_2m", "cloud_cover",
             "wind_speed_10m", "shortwave_radiation"]
linhas_regressao = []
for indice, data_hora in enumerate(horas_api["time"]):
    hora = int(data_hora[11:13])
    valores = [horas_api[nome][indice] for nome in nomes_api]
    if 7 <= hora <= 17 and all(valor is not None for valor in valores):
        linhas_regressao.append([data_hora, *valores[:4], hora, valores[4]])
if len(linhas_regressao) < 100:
    raise ValueError("Poucos horários válidos: confira a resposta do Open-Meteo")
print("Horas válidas:", len(linhas_regressao))
print("Primeiro exemplo:", linhas_regressao[0])

In [ ]:
with open("meteo_regressao_orange.csv", "w", newline="", encoding="utf-8-sig") as arquivo:
    escritor = csv.writer(arquivo)
    escritor.writerow(["data_hora", "temperatura_c", "umidade_pct",
                       "nuvens_pct", "vento_kmh", "hora", "radiacao_w_m2"])
    escritor.writerows(linhas_regressao)
print("Salvo: meteo_regressao_orange.csv")

### O que você deve fazer — regressão

1. Carregue o CSV, explique as colunas, examine valores ausentes e apresente ao menos uma visualização. Defina `X` (cinco entradas) e `y` (`radiacao_w_m2`).
2. Mantenha as linhas em ordem de `data_hora`: use aproximadamente **80% das primeiras horas para treino** e as 20% finais para teste. Evite embaralhar as horas. Ajuste transformações apenas com o treino.
3. Escolha, justifique e treine **três algoritmos de regressão diferentes** com a mesma divisão. Os imports e a implementação são sua responsabilidade.
4. Compare **MAE** (W/m²), **MSE** ((W/m²)²) e **R²**; faça um gráfico de valores reais × previstos de pelo menos um modelo.
5. Interprete os erros, discuta o peso da hora e explique por que essa estimativa da radiação **não prevê automaticamente a energia produzida** por um sistema fotovoltaico.

**O que registrar:** tabela comparativa dos três algoritmos, gráfico e interpretação escrita. Nenhum valor de `radiacao_w_m2` deve ser usado para criar uma entrada do mesmo registro.

## Solução da Tarefa 2 — Regressão da radiação solar

### 2.1 Carregar e explorar o CSV

In [ ]:
meteo = pd.read_csv("meteo_regressao_orange.csv", encoding="utf-8-sig", parse_dates=["data_hora"])
meteo = meteo.sort_values("data_hora").reset_index(drop=True)
print("Linhas e colunas:", meteo.shape)
print("Período:", meteo["data_hora"].min(), "a", meteo["data_hora"].max())
print("Horas presentes:", sorted(meteo["hora"].unique()))
print("\nValores ausentes por coluna:")
print(meteo.isna().sum())
meteo.head()

In [ ]:
meteo.drop(columns="data_hora").describe().T

In [ ]:
fig, eixos = plt.subplots(2, 2, figsize=(16, 9))

eixos[0, 0].plot(meteo["data_hora"], meteo["radiacao_w_m2"], lw=0.6)
eixos[0, 0].set_title("Radiação ao longo do período (7h–17h)")
eixos[0, 0].set_ylabel("radiação (W/m²)")

sns.boxplot(data=meteo, x="hora", y="radiacao_w_m2", ax=eixos[0, 1], color="#f2b134")
eixos[0, 1].set_title("Radiação por hora local")
eixos[0, 1].set_ylabel("radiação (W/m²)")

sns.scatterplot(data=meteo, x="nuvens_pct", y="radiacao_w_m2", hue="hora",
                palette="viridis", s=12, ax=eixos[1, 0])
eixos[1, 0].set_title("Cobertura de nuvens × radiação")

sns.heatmap(meteo.drop(columns="data_hora").corr(), annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, ax=eixos[1, 1])
eixos[1, 1].set_title("Correlação de Pearson")
plt.tight_layout()
plt.show()

*Interpretação `t2_exploracao` pendente: executar o notebook com os dados das APIs.*

### 2.2 Definir `X` e `y` e separar por tempo

- `X` = `temperatura_c`, `umidade_pct`, `nuvens_pct`, `vento_kmh`, `hora` (cinco entradas).
- `y` = `radiacao_w_m2`.
- `data_hora` serve apenas para ordenar e separar; não entra em `X`. Nenhuma entrada é derivada de `radiacao_w_m2`.

As linhas estão ordenadas por `data_hora`: as **primeiras 80% das horas** formam o treino e as **últimas 20%** formam o teste, **sem embaralhar**. Assim o modelo é avaliado em dias posteriores aos que viu no treino.

In [ ]:
entradas = ["temperatura_c", "umidade_pct", "nuvens_pct", "vento_kmh", "hora"]
X_reg = meteo[entradas]
y_reg = meteo["radiacao_w_m2"]

corte = int(len(meteo) * 0.80)
X_treino_r, X_teste_r = X_reg.iloc[:corte], X_reg.iloc[corte:]
y_treino_r, y_teste_r = y_reg.iloc[:corte], y_reg.iloc[corte:]
datas_teste = meteo["data_hora"].iloc[corte:]

print(f"Treino: {len(X_treino_r)} horas ({meteo['data_hora'].iloc[0]} a {meteo['data_hora'].iloc[corte - 1]})")
print(f"Teste:  {len(X_teste_r)} horas ({meteo['data_hora'].iloc[corte]} a {meteo['data_hora'].iloc[-1]})")

### 2.3 Três regressores

| Modelo | Por que foi escolhido | Pré-processamento |
|---|---|---|
| **Regressão Linear** | Referência simples e interpretável; mostra o quanto uma relação linear explica | `StandardScaler` (ajustado só no treino) |
| **Random Forest Regressor** (300 árvores) | Capta a curva não linear da radiação ao longo do dia e interações hora × nuvens | Nenhum |
| **Gradient Boosting** (`HistGradientBoostingRegressor`) | Árvores em sequência que corrigem os erros anteriores; costuma ser forte em dados tabulares | Nenhum |

A `hora` entra como número (7 a 17). Para a Regressão Linear isso impõe uma relação linear com a hora, enquanto a radiação tem formato de "sino" ao longo do dia — uma limitação que as árvores não têm. Um `DummyRegressor` (média do treino) aparece só como linha de base.

In [ ]:
regressores = {
    "Regressão Linear": make_pipeline(StandardScaler(), LinearRegression()),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=3, random_state=SEMENTE, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05, random_state=SEMENTE),
}

base_reg = DummyRegressor(strategy="mean").fit(X_treino_r, y_treino_r)
print("Linha de base (média do treino) — MAE no teste:",
      round(mean_absolute_error(y_teste_r, base_reg.predict(X_teste_r)), 1), "W/m²")

resultados_reg = []
previsoes_reg = {}
for nome, modelo in regressores.items():
    modelo.fit(X_treino_r, y_treino_r)
    previsto = modelo.predict(X_teste_r)
    previsoes_reg[nome] = previsto
    resultados_reg.append({
        "Algoritmo": nome,
        "MAE (W/m²)": mean_absolute_error(y_teste_r, previsto),
        "MSE ((W/m²)²)": mean_squared_error(y_teste_r, previsto),
        "R²": r2_score(y_teste_r, previsto),
    })

tabela_reg = pd.DataFrame(resultados_reg).set_index("Algoritmo").sort_values("MAE (W/m²)")
print("Avaliação: divisão temporal — primeiras 80% das horas no treino, últimas 20% no teste (sem embaralhar)")
tabela_reg

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(18, 5.5), sharex=True, sharey=True)
limite = max(y_teste_r.max(), max(p.max() for p in previsoes_reg.values())) * 1.05
for eixo, (nome, previsto) in zip(eixos, previsoes_reg.items()):
    eixo.scatter(y_teste_r, previsto, s=12, alpha=0.6)
    eixo.plot([0, limite], [0, limite], color="red", lw=1, label="previsão perfeita")
    eixo.set_title(f"{nome}  (R² = {r2_score(y_teste_r, previsto):.3f})")
    eixo.set_xlabel("radiação real (W/m²)")
    eixo.set_xlim(0, limite)
    eixo.set_ylim(0, limite)
eixos[0].set_ylabel("radiação prevista (W/m²)")
eixos[0].legend()
plt.suptitle("Valores reais × previstos no conjunto de teste")
plt.tight_layout()
plt.show()

In [ ]:
melhor_reg = tabela_reg.index[0]
dias = datas_teste.dt.normalize().unique()[:7]
janela = datas_teste.dt.normalize().isin(dias).values

fig, eixo = plt.subplots(figsize=(16, 4.5))
x_pos = np.arange(janela.sum())
eixo.plot(x_pos, y_teste_r.values[janela], marker="o", ms=3, label="real")
eixo.plot(x_pos, previsoes_reg[melhor_reg][janela], marker="o", ms=3, label=f"previsto — {melhor_reg}")
eixo.set_xticks(x_pos[::11])
eixo.set_xticklabels(datas_teste[janela].dt.strftime("%d/%m").values[::11])
eixo.set_ylabel("radiação (W/m²)")
eixo.set_title("Primeiros 7 dias do teste, horas 7h–17h em sequência")
eixo.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Erro absoluto médio por hora do dia no conjunto de teste (melhor modelo)
erro_hora = (pd.DataFrame({"hora": X_teste_r["hora"].values,
                           "erro_abs": np.abs(y_teste_r.values - previsoes_reg[melhor_reg])})
             .groupby("hora")["erro_abs"].mean())
print("MAE por hora —", melhor_reg)
erro_hora.to_frame("MAE (W/m²)")

In [ ]:
# Importância da hora: queda de desempenho do Random Forest quando `hora` é retirada de X
sem_hora = [c for c in entradas if c != "hora"]
rf_sem_hora = RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                    random_state=SEMENTE, n_jobs=-1)
rf_sem_hora.fit(X_treino_r[sem_hora], y_treino_r)
previsto_sem_hora = rf_sem_hora.predict(X_teste_r[sem_hora])
pd.DataFrame({
    "MAE (W/m²)": [tabela_reg.loc["Random Forest", "MAE (W/m²)"], mean_absolute_error(y_teste_r, previsto_sem_hora)],
    "R²": [tabela_reg.loc["Random Forest", "R²"], r2_score(y_teste_r, previsto_sem_hora)],
}, index=["Random Forest com hora", "Random Forest sem hora"])

### 2.4 Interpretação — regressão

*Interpretação `t2_conclusao` pendente: executar o notebook com os dados das APIs.*

## Resumo final das seis comparações

In [ ]:
print("TAREFA 1 — Classificação (holdout estratificado 80/20, random_state=42; métricas macro)")
display(tabela_clf.round(3))
print("TAREFA 2 — Regressão (divisão temporal 80/20, sem embaralhar)")
display(tabela_reg.round(3))

# Entrega pelo GitHub

Envie **apenas o link do repositório público**. Organize-o com:

- `README.md` contendo objetivo, fontes e período dos dados, instruções para executar o notebook e resumo dos resultados;
- notebook `.ipynb` com a consulta às APIs, sua análise, os **três classificadores e os três regressores**, todos os imports necessários adicionados por você e conclusões em Markdown;
- os dois CSVs gerados (ou instruções completas e testadas para reproduzi-los) e eventuais figuras utilizadas.

Verifique se o notebook roda **na ordem das células** a partir de um ambiente limpo. Não inclua senhas, tokens ou arquivos sem relação com a avaliação. A banca deve conseguir encontrar as seis comparações e as respostas às duas tarefas sem procurar uma agulha no palheiro.

### Fontes

- [ANEEL — SIGA](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel)
- [ANEEL — recurso e campos usados](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel/resource/11ec447d-698d-4ab8-977f-b424d5deee6a)
- [Open-Meteo — histórico e unidades](https://open-meteo.com/en/docs/historical-weather-api)